# Submission B: A with its 9 most uncertain rows flipped

| | |
|---|---|
| Submitted file | `sub_B_hedge.csv` |
| Submitted (UTC) | 2026-10-07 10:29 |
| Public score | 0.95580 (173/181) |
| Rule | A with its 9 most uncertain rows flipped |

B is A with the m most uncertain test rows flipped; m is chosen by Monte Carlo to maximise the expected best-of-two private score, with each row's probability taken from the sharp band's out-of-fold error rate at that distance from the edge. The rows are picked by this algorithm, not by hand.

**Run it:** on Kaggle, upload this notebook and add the competition data (*Add Input → Competitions → The Great Kacchi Aloo Mystery*), then *Save & Run All* and submit `submission.csv`. Locally, run it from this folder (it reads `../data/train.csv` and `../data/test.csv`). The last cell checks the output against the file that was actually submitted.

In [1]:
import glob, hashlib, os, unicodedata
import numpy as np, pandas as pd

found = sorted(glob.glob("/kaggle/input/**/train.csv", recursive=True)) + [p for p in ("../data/train.csv", "data/train.csv") if os.path.exists(p)]
root = os.path.dirname(found[0])
tr_raw = pd.read_csv(f"{root}/train.csv", dtype=str)
te_raw = pd.read_csv(f"{root}/test.csv", dtype=str)


def to_number(v):
    """Float from a cell; any Unicode decimal digit (e.g. Bangla) becomes ASCII."""
    if pd.isna(v):
        return np.nan
    s = "".join(str(unicodedata.decimal(c)) if unicodedata.decimal(c, None) is not None else c
                for c in str(v).strip().replace(",", ""))
    try:
        return float(s)
    except ValueError:
        return np.nan


def add_apg(d):
    """aloo per guest; the few aloo counts typed x10 (ratio > 3) are divided by 10."""
    guests, aloo = d.guests.map(to_number), d.aloo_count.map(to_number)
    aloo = np.where(aloo / guests > 3.0, aloo / 10, aloo)
    return d.assign(apg=aloo / guests)


tr = add_apg(tr_raw).dropna(subset=["apg"]).reset_index(drop=True)   # 24 train rows have no aloo_count
tr["y"] = tr.went_back_for_seconds.astype(int)
te = add_apg(te_raw)
print(f"data: {root} | train {len(tr)} rows with aloo_count | test {len(te)} rows")

data: ../data | train 776 rows with aloo_count | test 400 rows


In [2]:
def midcut_cuts(r, y):
    """Each edge sits at the midpoint between neighbouring training ratios that maximises accuracy on its side of
    1.5 (median of tied optima)."""
    def best_cut(x, yy, side):
        xs = np.sort(np.unique(x))
        c = (xs[:-1] + xs[1:]) / 2
        accs = np.array([((x >= t) == yy).mean() if side == "lo" else ((x <= t) == yy).mean() for t in c])
        b = np.flatnonzero(accs == accs.max())
        return c[b[len(b) // 2]]
    m = r < 1.5
    return best_cut(r[m], y[m], "lo"), best_cut(r[~m], y[~m], "hi")


def band(x, lo, hi):
    return ((x >= lo) & (x <= hi)).astype(int)

In [3]:
from sklearn.model_selection import StratifiedKFold


def sharp_band_probabilities(tr, te, bins=(0.0, 0.01, 0.035, 0.1, 0.4, np.inf), repeats=10, seed=0):
    """P(y=1) per test row = the sharp band's out-of-fold error rate at that distance from the nearest edge.
    Test rows inside the training gap around an edge get P from a uniform prior on where the cut lies."""
    r, y = tr.apg.values, tr.y.values
    lo, hi = midcut_cuts(r, y)
    d_oof, e_oof = [], []
    for k in range(repeats):
        for t, v in StratifiedKFold(5, shuffle=True, random_state=seed + k).split(tr, y):
            l2, h2 = midcut_cuts(r[t], y[t])
            d_oof.append(np.minimum(abs(r[v] - l2), abs(r[v] - h2)))
            e_oof.append(band(r[v], l2, h2) != y[v])
    d_oof, e_oof = np.concatenate(d_oof), np.concatenate(e_oof)
    idx = np.digitize(d_oof, bins[1:-1])
    err = np.array([(e_oof[idx == b].sum() + 0.5) / ((idx == b).sum() + 1.0) for b in range(len(bins) - 1)])
    below_lo, above_lo = r[r < lo].max(), r[(r >= lo) & (r < 1.5)].min()
    below_hi, above_hi = r[(r <= hi) & (r > 1.5)].max(), r[r > hi].min()
    x = te.apg.values
    d = np.minimum(abs(x - lo), abs(x - hi))
    e = err[np.digitize(d, bins[1:-1])]
    P = np.where(band(x, lo, hi) == 1, 1 - e, e)
    q_lo = np.clip((x - below_lo) / (above_lo - below_lo), 0, 1)
    q_hi = np.clip((above_hi - x) / (above_hi - below_hi), 0, 1)
    g_lo, g_hi = (x > below_lo) & (x < above_lo), (x > below_hi) & (x < above_hi)
    P = np.where(g_lo, q_lo * (1 - err[0]) + (1 - q_lo) * err[0], P)
    P = np.where(g_hi, q_hi * (1 - err[0]) + (1 - q_hi) * err[0], P)
    return P, (lo, hi), err


P, (LO, HI), err = sharp_band_probabilities(tr, te)
print(f"cut-points {LO:.5f} / {HI:.5f}; out-of-fold error rate by distance to the edge "
      f"(0-0.01, 0.01-0.035, 0.035-0.1, 0.1-0.4, >0.4): {np.round(err, 3)}")

cut-points 0.97068 / 2.00983; out-of-fold error rate by distance to the edge (0-0.01, 0.01-0.035, 0.035-0.1, 0.1-0.4, >0.4): [0.127 0.374 0.148 0.022 0.   ]


In [4]:
def hedge_pair(p, tiebreak, n_public=181, sims=20000, seed=0, max_flip=40):
    """A = p > 0.5. B = A with its m most uncertain rows flipped (|p - 0.5| ascending, ties by distance to the
    nearest edge). m maximises E[max(private correct of A, of B)] over simulated labels y_i ~ Bernoulli(p_i)
    and random 181-row public splits."""
    rng = np.random.default_rng(seed)
    n = len(p)
    A = (p > 0.5).astype(int)
    order = np.lexsort((tiebreak, np.abs(p - 0.5)))
    Y = rng.random((sims, n)) < p[None, :]
    priv = np.ones((sims, n), bool)
    for s in range(sims):
        priv[s, rng.choice(n, n_public, replace=False)] = False
    accA = ((Y == A[None, :]) & priv).sum(1)
    gain = {}
    for m in range(max_flip + 1):
        B = A.copy(); B[order[:m]] = 1 - B[order[:m]]
        gain[m] = float(np.maximum(accA, ((Y == B[None, :]) & priv).sum(1)).mean() - accA.mean())
    m_best = max(gain, key=gain.get)
    B = A.copy(); B[order[:m_best]] = 1 - B[order[:m_best]]
    return A, B, m_best, gain


d_edge = np.minimum(abs(te.apg.values - LO), abs(te.apg.values - HI))
A, pred, m, gain = hedge_pair(P, d_edge)
assert (A == band(te.apg.values, LO, HI)).all(), "A must equal the sharp band"
print(f"m = {m} rows flipped; expected best-of-two private gain over A alone: {gain[m]:+.3f} rows")
print(te.assign(P=P.round(3), A=A, B=pred).loc[A != pred, ["wedding_id", "guests", "aloo_count", "apg", "P", "A", "B"]]
      .sort_values("apg").to_string(index=False))

m = 9 rows flipped; expected best-of-two private gain over A alone: +0.423 rows
wedding_id guests aloo_count      apg     P  A  B
     W1006    269        257 0.955390 0.374  0  1
     W1163    203        197 0.970443 0.384  0  1
     W0968    578        561 0.970588 0.456  0  1
     W1119    426        418 0.981221 0.626  1  0
     W1157    317        313 0.987382 0.626  1  0
     W0839    376        750 1.994681 0.626  1  0
     W1079    374        756 2.021390 0.374  0  1
     W1023    935       1894 2.025668 0.374  0  1
     W0974    442        896 2.027149 0.374  0  1


In [5]:
sub = pd.DataFrame({"wedding_id": te.wedding_id, "went_back_for_seconds": np.asarray(pred).astype(int)})
sub.to_csv("submission.csv", index=False)
fp = hashlib.md5("".join(f"{i}{p}" for i, p in zip(sub.wedding_id, sub.went_back_for_seconds)).encode()).hexdigest()
print(f"wrote submission.csv: {sub.went_back_for_seconds.sum()} ones / {len(sub)} rows")
print(f"identical to the submitted file 'sub_B_hedge.csv': {fp == '06a0392eb63dbaedf0ae11c9160f863a'}")

wrote submission.csv: 185 ones / 400 rows
identical to the submitted file 'sub_B_hedge.csv': True
